In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program to calculate the categorical cross-entropy loss for a batch of predictions.
  Given a matrix of predicted logits $Z$ of size $N \times C$ and a vector of true class labels <code>true_labels</code> of size $N$, compute the average cross-entropy loss over the batch.
  The loss for a single sample $j$ with logits $z_j = [z_{j1}, \ldots, z_{jC}]$ and true label $y_j$ is calculated using the numerically stable formula:
  $$ \text{Loss}_j = \log\left(\sum_{k=1}^{C} e^{z_{jk}}\right) - z_{j, y_j} $$
  The final output stored in the <code>loss</code> variable should be the average loss over the $N$ samples:
  $$ L = \frac{1}{N} \sum_{j=1}^{N} \text{Loss}_j $$
  The input parameters are <code>logits</code>, <code>true_labels</code>, <code>N</code> (number of samples), and <code>C</code> (number of classes). The result should be stored in <code>loss</code> (a pointer to a single float).
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result (average loss) must be stored in <code>loss</code></li>
</ul>

<h2>Example 1:</h2>
<pre>Input:  N = 2, C = 3
        logits = [[1.0, 2.0, 0.5], [0.1, 3.0, 1.5]]
        true_labels = [1, 1]
Output: loss = [0.3548926]</pre>


<h2>Example 2:</h2>
<pre>Input:  N = 3, C = 4
        logits = [[-0.5, 1.5, 0.0, 1.0], [2.0, -1.0, 0.5, 0.5], [0.0, 0.0, 0.0, 0.0]]
        true_labels = [3, 0, 1]
Output: loss = [0.98820376]</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 10,000</li>
  <li>2 &le; <code>C</code> &le; 1,000</li>
  <li>-10.0 &le; <code>logits[i, j]</code> &le; 10.0</li>
  <li>0 &le; <code>true_labels[i]</code> &le; <code>C</code></li>

  <li>Performance is measured with <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// logits, true_labels, loss are device pointers
extern "C" void solve(const float* logits, const int* true_labels, float* loss, int N, int C) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# logits, true_labels, loss are tensors on the GPU
@cute.jit
def solve(
    logits: cute.Tensor, true_labels: cute.Tensor, loss: cute.Tensor, N: cute.Int32, C: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# logits, true_labels are tensors on the GPU
@jax.jit
def solve(logits: jax.Array, true_labels: jax.Array, N: int, C: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    logits: UnsafePointer[Float32, MutExternalOrigin],
    true_labels: UnsafePointer[Int32, MutExternalOrigin],
    loss: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    C: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# logits, true_labels, loss are tensors on the GPU
def solve(logits: torch.Tensor, true_labels: torch.Tensor, loss: torch.Tensor, N: int, C: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# logits, true_labels, loss are tensors on the GPU
def solve(logits: torch.Tensor, true_labels: torch.Tensor, loss: torch.Tensor, N: int, C: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/25_categorical_cross_entropy_loss/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
